# Coleta e preparação de notícias para PLN

**Etapa Prática 1 — Processamento de Linguagem Natural**

Fonte: [Blog do Jaime](https://blogdojaime.com.br/), portal de notícias de Blumenau e região.

Este notebook mostra o trabalho inteiro, passo a passo. Ele traz o mesmo código dos três
scripts do projeto (`coletar.py`, `preparar.py` e `analisar.py`), só que fatiado e explicado,
para dar para acompanhar o que acontece com o texto em cada etapa.

Ele roda sozinho: instala o que precisa na primeira célula e não depende de nenhum arquivo
do projeto.

| Parte | O que faz |
|---|---|
| 1 | Baixa as notícias do site |
| 2 | Tira os créditos e a propaganda do meio do texto |
| 3 | Normaliza e separa em palavras |
| 4 | Remove as palavras muito comuns |
| 5 | Reduz as palavras a radicais e lemas |
| 6 | Descarta notícias repetidas e curtas demais |
| 7 | Junta tudo num pipeline só |
| 8 | Analisa a base e confere a qualidade |
| 9 | Salva o resultado |

## Instalação

- `requests` e `beautifulsoup4`: baixar e ler o HTML das páginas
- `nltk`: cortar as palavras em radicais (stemming)
- `simplemma`: achar a palavra do dicionário (lematização)
- `matplotlib`: os gráficos

In [ ]:
%pip install --quiet requests beautifulsoup4 nltk simplemma matplotlib
print("Pronto.")

In [ ]:
import hashlib
import json
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime
from statistics import mean, median
from urllib.parse import urljoin, urlparse

import matplotlib.pyplot as plt
import requests
import simplemma
from bs4 import BeautifulSoup
from nltk.stem import SnowballStemmer

SITE = "https://blogdojaime.com.br"

# O site responde com erro 403 se a gente não se identificar como um navegador.
CABECALHOS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

print("Tudo importado.")

---
# Parte 1 — Baixar as notícias

De cada notícia queremos guardar seis coisas: **título, data, categoria, texto, endereço** e o
momento em que foi coletada.

Três coisas do site que deram trabalho e explicam o código:

1. **Ele bloqueia quem não parece navegador.** Sem o cabeçalho `User-Agent` acima, toda página
   volta com erro 403.
2. **A data e a categoria confiáveis estão escondidas.** No texto visível a data aparece de
   jeitos diferentes ("31 de agosto", "31/08/2026"), mas o WordPress publica um bloco
   `<script type="application/ld+json">` com a informação certinha. É de lá que a gente lê.
3. **O tema do site é o Elementor**, então o texto da notícia às vezes está numa `div` chamada
   `elementor-widget-theme-post-content` em vez da `entry-content` de sempre. O código tenta
   uma e depois a outra.

In [ ]:
def limpar_espacos(texto):
    """Troca quebras de linha e espaços repetidos por um espaço só."""
    return " ".join(texto.split())


def baixar_pagina(url, espera=1.5):
    """Baixa o HTML de uma página e espera um pouco antes da próxima.

    A espera é importante: as páginas antigas do site são lentas e, se a
    gente pedir tudo de uma vez, o site sai do ar.
    """
    resposta = requests.get(url, headers=CABECALHOS, timeout=30)
    resposta.raise_for_status()
    time.sleep(espera)
    return resposta.text


def ler_json_ld(sopa, chave):
    """Procura uma informação nos blocos JSON-LD da página."""
    for script in sopa.find_all("script", type="application/ld+json"):
        try:
            dados = json.loads(script.string or "")
        except json.JSONDecodeError:
            continue

        # O WordPress costuma colocar tudo dentro de uma lista chamada "@graph".
        if isinstance(dados, dict) and "@graph" in dados:
            blocos = dados["@graph"]
        elif isinstance(dados, list):
            blocos = dados
        else:
            blocos = [dados]

        for bloco in blocos:
            if not isinstance(bloco, dict):
                continue
            valor = bloco.get(chave)
            if isinstance(valor, list) and valor:
                valor = valor[0]
            if valor:
                return limpar_espacos(str(valor))
    return ""


print("Funções de leitura prontas.")

In [ ]:
def extrair_noticia(html, url):
    """Monta o dicionário de uma notícia a partir do HTML da página."""
    sopa = BeautifulSoup(html, "html.parser")

    titulo = sopa.find("h1")
    titulo = limpar_espacos(titulo.get_text(" ", strip=True)) if titulo else ""

    # Procura o texto da notícia. O primeiro seletor é o do tema atual do site;
    # os outros são para posts antigos.
    corpo = None
    for seletor in (".elementor-widget-theme-post-content", ".entry-content",
                    ".post-content", "article"):
        corpo = sopa.select_one(seletor)
        if corpo:
            break

    texto = ""
    if corpo:
        # Joga fora o que não é notícia: script, menu, botão de compartilhar.
        for lixo in corpo.select("script, style, nav, form, iframe, .social-share"):
            lixo.decompose()
        # O " " no get_text evita colar palavras quando tem <br> no parágrafo:
        # sem ele, "Data:<br>Hora:" viraria "Data:Hora:".
        paragrafos = [limpar_espacos(p.get_text(" ", strip=True)) for p in corpo.find_all("p")]
        texto = "\n".join(p for p in paragrafos if p)

    return {
        "titulo": titulo,
        "data": ler_json_ld(sopa, "datePublished"),
        "categoria": ler_json_ld(sopa, "articleSection"),
        "texto": texto,
        "url": url,
        "coletado_em": datetime.now().isoformat(timespec="seconds"),
    }


print("Extrator pronto.")

### Testando o extrator sem entrar na internet

Antes de sair baixando página, vale testar num HTML de mentira que imita o do site: com o
bloco JSON-LD, a `div` do Elementor e um `<br>` no meio do parágrafo. Assim dá para conferir
que o extrator faz o que a gente espera.

In [ ]:
HTML_DE_TESTE = """
<html><head>
<script type="application/ld+json">
{"@graph":[{"@type":"BlogPosting",
            "datePublished":"2026-05-12T09:15:00-03:00",
            "articleSection":"Ocorrências"}]}
</script></head><body>
<h1>Carreta destruiu parte do pórtico no bairro Testo Rega.</h1>
<div class="elementor-widget-theme-post-content">
  <p>Data: 11/05/2026<br>Hora: 19h27min</p>
  <p>Acidente com danos materiais na rua Presidente Costa e Silva.</p>
  <p>Fonte: Polícia Militar</p>
  <script>propaganda()</script>
</div></body></html>
"""

teste = extrair_noticia(HTML_DE_TESTE, "https://blogdojaime.com.br/teste/")

for campo in ("titulo", "data", "categoria"):
    print(f"{campo:>10}: {teste[campo]}")
print("\ntexto:")
for linha in teste["texto"].split("\n"):
    print("   ", linha)

# Se alguma coisa quebrar, o notebook para aqui em vez de seguir com dado errado.
assert teste["categoria"] == "Ocorrências"       # veio do JSON-LD
assert "19h27min" in teste["texto"]              # o <br> virou espaço
assert "propaganda" not in teste["texto"]        # o <script> foi removido
print("\nDeu certo: os campos saíram como esperado.")

### Achando os links das notícias

Uma página de categoria tem link para tudo: menu, outras categorias, próxima página, imagem.
Precisamos separar o que é notícia.

O jeito mais simples é olhar o formato do endereço. No site, notícia tem **um pedaço só**
depois do domínio (`/titulo-da-noticia/`). Categoria e paginação têm mais de um
(`/category/tempo/`, `/category/tempo/page/2/`).

In [ ]:
def eh_link_de_noticia(link):
    """Diz se o endereço é de uma notícia."""
    endereco = urlparse(link)
    if endereco.netloc.replace("www.", "") != "blogdojaime.com.br":
        return False  # link para fora do site

    caminho = endereco.path.strip("/")
    if not caminho or "/" in caminho:
        return False  # notícia tem um pedaço só
    if caminho in ("noticias", "contato", "anuncie-conosco"):
        return False  # páginas fixas do site
    if caminho.startswith(("category", "tag", "author", "page", "wp-")):
        return False  # listagens
    return not caminho.endswith((".jpg", ".jpeg", ".png", ".webp", ".pdf"))


def procurar_links(html, url_da_pagina):
    """Devolve os links de notícia de uma página de listagem, sem repetir."""
    sopa = BeautifulSoup(html, "html.parser")
    links = []
    for tag in sopa.find_all("a", href=True):
        link = urljoin(url_da_pagina, tag["href"]).split("#")[0].rstrip("/") + "/"
        if eh_link_de_noticia(link) and link not in links:
            links.append(link)
    return links


html_listagem = """<main>
  <a href='/category/tempo/'>Tempo</a>
  <a href='/chuva-volumosa-em-blumenau/'>31 de agosto 2026</a>
  <a href='/noticias/'>veja mais notícias</a>
  <a href='https://outro-site.com/materia'>site de fora</a>
  <a href='/foto.jpg'>uma imagem</a>
</main>"""

achados = procurar_links(html_listagem, f"{SITE}/category/tempo/")
print("links aceitos:", achados)
assert achados == ["https://blogdojaime.com.br/chuva-volumosa-em-blumenau/"]
print("Deu certo: sobrou só a notícia.")

### Coletando de verdade

Agora sim. Escolhemos poucas categorias e uma página de cada, para não demorar. No trabalho
completo rodamos `python coletar.py --paginas 12`, que leva mais de uma hora.

Se a internet falhar ou o site estiver fora do ar, a célula usa um conjunto de notícias de
exemplo, para o resto do notebook continuar funcionando.

In [ ]:
CATEGORIAS = {"ocorrencias": 1, "tempo": 1, "eventos": 1}

noticias = []
de_onde_veio = {}  # endereço da notícia -> categoria onde ela foi encontrada

for categoria, paginas in CATEGORIAS.items():
    print(f"[{categoria}]")
    for numero in range(1, paginas + 1):
        endereco = f"{SITE}/category/{categoria}/"
        if numero > 1:
            endereco += f"page/{numero}/"
        try:
            links = procurar_links(baixar_pagina(endereco), endereco)
        except Exception as erro:
            print(f"   não abriu: {erro}")
            continue
        for link in links:
            try:
                noticias.append(extrair_noticia(baixar_pagina(link), link))
                de_onde_veio[link] = categoria
            except Exception as erro:
                print(f"   {link} falhou: {erro}")
    print(f"   temos {len(noticias)} notícias até agora")

if not noticias:
    print("\nSem acesso ao site. Usando notícias de exemplo.")
    EXEMPLOS = [
        ("Acidente na rua Itajaí deixa dois feridos em Blumenau.", "Ocorrências",
         "Data: 11/05/2026 Hora: 19h27min\nO condutor, do sexo masculino, 34 anos, perdeu o "
         "controle e colidiu contra o muro. Os bombeiros atenderam a ocorrência e "
         "encaminharam as vítimas ao hospital.\nFonte: Polícia Militar"),
        ("Sol e frio de 14°C em Blumenau nesta quinta-feira.", "Tempo",
         "28/05/2026 A massa de ar seco e frio segue atuando sobre Santa Catarina, mantendo "
         "o tempo firme e ensolarado em Blumenau. Máxima entre 21 e 23ºC.\nFonte: Defesa Civil"),
        ("Prefeitura amplia programação dos 176 anos da cidade.", "Eventos",
         "A prefeitura divulgou novas atrações para a semana de aniversário da cidade. "
         "A programação inclui música, teatro e feira de artesanato no centro.\n"
         "Foto: Giovanni Silva/PMB"),
        ("Obras na rua Frederico Jensen mudam o trânsito.", "Trânsito",
         "A reurbanização começa nesta semana e provoca mudanças no trânsito. Motoristas "
         "devem usar rotas alternativas durante as obras.\nFonte: Prefeitura de Blumenau"),
        ("Vacinação contra a gripe segue nas unidades de saúde.", "Saúde",
         "As unidades de saúde da Velha e da Itoupava seguem com vacinação contra a gripe. "
         "O atendimento acontece das 8h às 17h.\nFonte: Secretaria de Saúde"),
    ]
    agora = datetime.now().isoformat(timespec="seconds")
    for i, (titulo, categoria, texto) in enumerate(EXEMPLOS):
        endereco = f"{SITE}/exemplo-{i}/"
        noticias.append({"titulo": titulo, "data": f"2026-0{i + 5}-1{i}T09:00:00-03:00",
                         "categoria": categoria, "texto": texto, "url": endereco,
                         "coletado_em": agora})
        de_onde_veio[endereco] = categoria.lower()

print(f"\n{len(noticias)} notícias coletadas.\n")
for n in noticias[:3]:
    print(f"  [{n['categoria']}] {n['titulo'][:64]}")

---
# Parte 2 — Tirar os créditos e a propaganda

## O problema

No HTML, o crédito da foto e a assinatura da fonte estão nos mesmos parágrafos que a notícia.
Não dá para separar olhando só a estrutura da página. Então, depois de extrair o texto, sobra
coisa que não é notícia:

- `Fonte: Prefeitura de Blumenau`, `Foto: Giovanni Silva/PMB`
- convite para mandar WhatsApp para o Jaime, `#blogdojaime`, "foto meramente ilustrativa"

Isso não é detalhe. Na base completa, **`fonte` era a 5ª palavra mais comum**, com 680
aparições. Parecia vocabulário de jornalismo, mas era rodapé.

## Como decidimos o que cortar

Olhando a base antes de escrever a regra. Das **455** linhas que começavam com `Fonte:`,
**454 estavam nas duas últimas linhas** do texto. Ou seja: é assinatura no fim, não conteúdo.
Por isso a gente pode apagar a linha inteira sem medo.

## O cuidado que a gente tomou

O texto original **nunca** é apagado. A versão limpa vai para um campo novo, `texto_limpo`,
para dar para comparar e voltar atrás se precisar.

In [ ]:
# Linha que COMEÇA com um desses rótulos é crédito, não notícia.
CREDITO_NO_INICIO = re.compile(r"^\s*(fonte|foto|imagem|crédito|credito|oferecimento)s?\s*:", re.I)

# Às vezes o crédito vem grudado no fim de uma frase de verdade.
CREDITO_NO_FIM = re.compile(r"\s*\b(fonte|crédito|credito|oferecimento)s?\s*:.*$", re.I)

# Chamadas do próprio blog.
PROPAGANDA = re.compile(
    r"mande um whatsapp para o jaime|foto meramente ilustrativa|#blogdojaime", re.I
)


def limpar_creditos(texto):
    """Tira os créditos de fonte e foto e as chamadas do blog."""
    linhas_boas = []
    for linha in (texto or "").split("\n"):
        if PROPAGANDA.search(linha) or CREDITO_NO_INICIO.match(linha):
            continue  # linha inteira é crédito
        linha = CREDITO_NO_FIM.sub("", linha).strip()  # crédito no fim da frase
        if linha:
            linhas_boas.append(linha)
    return "\n".join(linhas_boas)


exemplo = (
    "O trânsito ficará lento durante as obras.\n"
    "Sentimentos aos familiares e amigos. Fonte: Central Funerária. Oferecimento: Jardim.\n"
    "Fonte: Prefeitura de Blumenau\n"
    "Foto: Giovanni Silva/PMB"
)

print("ANTES:")
for linha in exemplo.split("\n"):
    print("   ", linha)
print("\nDEPOIS:")
for linha in limpar_creditos(exemplo).split("\n"):
    print("   ", linha)

# Cuidado importante: "fonte" só é crédito quando vem seguida de dois-pontos.
assert limpar_creditos("A fonte do Parque Ramiro foi reformada.") == (
    "A fonte do Parque Ramiro foi reformada."
)
print("\nA palavra 'fonte' em uso normal continua no texto.")

---
# Parte 3 — Normalizar e separar em palavras

## Normalizar

Três coisas a gente faz, e uma que a gente **não** faz de propósito:

| O que | Por quê |
|---|---|
| deixar tudo minúsculo | "Blumenau" e "blumenau" têm que ser a mesma palavra |
| juntar espaços repetidos | quebra de linha e tabulação viram espaço simples |
| arrumar o Unicode (NFC) | a mesma letra acentuada pode ser gravada de dois jeitos |
| **não tirar os acentos** | "e" e "é" são palavras diferentes; tirar acento misturaria tudo |

## Separar em palavras

A expressão regular aceita dois casos: palavra (com acento, hífen e apóstrofo) e número (com
vírgula ou ponto). Como a pontuação não se encaixa em nenhum dos dois, **ela some sozinha** —
não precisa de um passo separado para tirar vírgula e ponto.

Uma coisa que descobrimos testando: o hífen só junta letra com letra. Então `terça-feira` fica
inteiro, mas `BR-470` vira `br` e `470`. Para contar palavra não faz diferença; para quem for
procurar nomes de rodovia depois, faz.

In [ ]:
PALAVRAS_E_NUMEROS = re.compile(r"[^\W\d_]+(?:[-'][^\W\d_]+)*|\d+(?:[.,]\d+)*", re.UNICODE)


def normalizar(texto):
    """Deixa minúsculo, arruma os espaços e mantém os acentos."""
    texto = unicodedata.normalize("NFC", texto or "")
    return " ".join(texto.split()).lower()


def tokenizar(texto):
    """Separa em palavras e números. A pontuação fica de fora."""
    return PALAVRAS_E_NUMEROS.findall(texto)


frase = "  TRÂNSITO na BR-470:\n chuva de 12,5 mm afetou 3 bairros!  "
print("como veio    :", repr(frase))
print("normalizado  :", repr(normalizar(frase)))
print("em palavras  :", tokenizar(normalizar(frase)))

print("\no que o hífen faz:")
for palavra in ("terça-feira", "guarda-civil", "BR-470", "covid-19"):
    print(f"   {palavra:<14} -> {tokenizar(normalizar(palavra))}")

assert "12,5" in tokenizar(normalizar(frase))        # número decimal fica inteiro
assert "trânsito" in tokenizar(normalizar(frase))    # o acento continua lá
assert ":" not in "".join(tokenizar(normalizar(frase)))  # a pontuação sumiu
print("\nNúmero e acento preservados, pontuação removida.")

---
# Parte 4 — Tirar as palavras muito comuns (stopwords)

"de", "que", "em", "para" aparecem em toda notícia e não dizem nada sobre o assunto. Se a
gente contar as palavras mais comuns sem tirar essas, o resultado é só isso e nada de útil.

A lista é a do NLTK para português: artigos, preposições, pronomes e as formas dos verbos
*ser*, *estar*, *ter* e *haver*. Acrescentamos algumas contrações que aparecem muito em
notícia (`nesta`, `neste`, `desta`).

**Escrevemos a lista aqui no código em vez de usar `nltk.download('stopwords')`** para o
programa rodar sem internet e dar sempre o mesmo resultado.

**Cuidado com o `não`:** ele está na lista do NLTK e a gente manteve. Isso quer dizer que
"não houve feridos" e "houve feridos" ficam iguais depois do filtro. Para quem for analisar
sentimento depois, isso é um problema — por isso guardamos também a lista completa de palavras
no campo `tokens`, sem filtro nenhum.

In [ ]:
PALAVRAS_VAZIAS = frozenset("""
a à ao aos aquela aquelas aquele aqueles aquilo as às até com como da das de dela delas
dele deles depois do dos e é ela elas ele eles em entre era eram éramos essa essas esse
esses esta está estamos estão estar estas estava estavam estávamos este esteja estejam
estejamos estes esteve estive estivemos estiver estivera estiveram estivéramos estiverem
estivermos estivesse estivessem estivéssemos estou eu foi fomos for fora foram fôramos
forem formos fosse fossem fôssemos fui há haja hajam hajamos hão havemos haver hei houve
houvemos houver houvera houverá houveram houvéramos houverão houverei houverem houveremos
houveria houveriam houveríamos houvermos houvesse houvessem houvéssemos isso isto já lhe
lhes mais mas me mesmo meu meus minha minhas muito na não nas nem no nos nós nossa nossas
nosso nossos num numa o os ou para pela pelas pelo pelos por qual quando que quem são se
seja sejam sejamos sem ser será serão serei seremos seria seriam seríamos seu seus só
somos sou sua suas também te tem tém temos tenha tenham tenhamos tenho terá terão terei
teremos teria teriam teríamos teu teus teve tinha tinham tínhamos tive tivemos tiver
tivera tiveram tivéramos tiverem tivermos tivesse tivessem tivéssemos tu tua tuas um uma
você vocês vos
nesta neste nestas nestes nessa nesse nessas nesses desta deste destas destes dessa desse
dessas desses naquela naquele daquela daquele nums numas
""".split())


def tirar_palavras_vazias(tokens):
    """Remove as palavras da lista acima."""
    return [t for t in tokens if t not in PALAVRAS_VAZIAS]


print(f"A lista tem {len(PALAVRAS_VAZIAS)} palavras.\n")

frase = normalizar("Nesta terça-feira não houve feridos no acidente da BR-470 em Blumenau")
todas = tokenizar(frase)
filtradas = tirar_palavras_vazias(todas)

print("todas as palavras :", todas)
print("depois do filtro  :", filtradas)
print(f"\nde {len(todas)} para {len(filtradas)} palavras")
print("Repare que o 'não' sumiu: a frase perdeu a negação.")

---
# Parte 5 — Radicais e lemas

Duas formas de juntar palavras parecidas numa só. Servem para o programa entender que
"chuva", "chuvas" e "chuvoso" falam da mesma coisa.

| | Radical (stemming) | Lema (lematização) |
|---|---|---|
| como funciona | corta o final da palavra | procura a palavra no dicionário |
| resultado | pode não existir (`notíc`) | é sempre palavra de verdade (`notícia`) |
| com verbo | `foram` vira `for` | `foram` vira `ser` |
| velocidade | bem rápido | mais devagar |

## Por que escolhemos essas duas bibliotecas

**Snowball, e não RSLP.** O RSLP foi feito especialmente para o português e seria a melhor
escolha, mas ele precisa de um arquivo extra baixado com `nltk.download('rslp')`. O Snowball
já vem junto com o NLTK e funciona offline.

**simplemma, e não spaCy.** O spaCy precisa baixar um modelo de uns 15 MB. O simplemma já vem
com o dicionário dentro. Em compensação ele erra em algumas palavras, como dá para ver abaixo.

In [ ]:
radicalizador = SnowballStemmer("portuguese")


def achar_radicais(tokens):
    """Corta o final das palavras: 'notícias' vira 'notíc'."""
    return [radicalizador.stem(t) for t in tokens]


def achar_lemas(tokens):
    """Acha a palavra do dicionário: 'chuvas' vira 'chuva'."""
    return [simplemma.lemmatize(t, lang="pt") for t in tokens]


exemplos = ["notícias", "chuvas", "corredores", "atendimento", "foram", "melhores", "bombeiros"]

print(f"{'palavra':<14}{'radical':<14}{'lema':<14}")
print("-" * 42)
for palavra, radical, lema in zip(exemplos, achar_radicais(exemplos), achar_lemas(exemplos)):
    print(f"{palavra:<14}{radical:<14}{lema:<14}")

print("\nO que dá para aprender com esses casos:")
print("  notícias -> 'notíc' não é palavra nenhuma, mas serve para comparar textos.")
print("  foram    -> o lema achou o verbo 'ser'; o radical só cortou o final.")
print("  melhores -> o lema deu 'melhorar', que está ERRADO ('melhores' é adjetivo,")
print("              o certo seria 'melhor'). Acontece porque o simplemma não olha")
print("              se a palavra é verbo ou adjetivo.")

---
# Parte 6 — Tirar notícias repetidas e curtas demais

## Repetidas

O site **republica a mesma notícia em dias seguidos**, mudando só o endereço (que termina em
`-2`, `-3`, `-4`). Acontece muito com nota de serviço: horário de posto de saúde, programação
de festa. Comparar o endereço não resolve, porque o endereço é diferente.

A solução é comparar o **conteúdo**. A gente gera um código (hash) a partir do título mais o
texto, depois de tirar acento, pontuação e maiúscula. Aí "Ação" e "ACAO!" geram o mesmo código
e a gente descobre que é a mesma notícia. Na base completa isso pegou 129 repetições.

## Curtas demais

Alguns posts são só uma imagem ou um vídeo: o parágrafo existe mas está vazio, e o texto sai
com zero palavra. Não é erro do nosso código, é o formato do post. Tiramos quem tem menos de
20 palavras.

In [ ]:
def gerar_chave(titulo, texto):
    """Cria um código do conteúdo, para achar notícia repetida.

    Tira acento, pontuação e maiúscula de propósito: assim duas versões da
    mesma notícia, escritas de jeito um pouco diferente, geram o mesmo código.
    """
    junto = f"{titulo} {texto}"
    sem_acento = unicodedata.normalize("NFKD", junto).encode("ascii", "ignore").decode()
    limpo = re.sub(r"[^a-z0-9]+", " ", sem_acento.lower()).strip()
    return hashlib.sha256(limpo.encode("utf-8")).hexdigest()


# Duas escritas da mesma notícia geram o mesmo código:
a = gerar_chave("Ação na Velha", "O time venceu!")
b = gerar_chave("ACAO NA VELHA", "o time venceu")
print("mesma notícia escrita diferente dá o mesmo código?", a == b)
assert a == b

def tirar_repetidas_e_curtas(lista, minimo_palavras=20):
    """Fica só com as notícias novas e com texto de tamanho razoável."""
    boas = []
    ja_vistas = set()
    repetidas = curtas = 0

    for noticia in lista:
        if noticia["palavras"] < minimo_palavras:
            curtas += 1
        elif noticia["chave"] in ja_vistas:
            repetidas += 1
        else:
            ja_vistas.add(noticia["chave"])
            boas.append(noticia)

    return boas, repetidas, curtas


print("Funções prontas (vamos usar na Parte 7).")

---
# Parte 7 — Juntando tudo

Agora a gente aplica tudo em cada notícia. A regra que seguimos:

> **Nada do original é apagado.** Cada etapa cria um campo novo. O `titulo` e o `texto`
> continuam exatamente como vieram do site.

Isso importa porque cada tarefa precisa de uma versão diferente do texto. Quem for procurar
nomes de pessoas e lugares precisa do texto original, com maiúscula e pontuação. Quem for
classificar ou fazer busca funciona melhor com o texto reduzido.

In [ ]:
def preparar_noticia(noticia):
    """Acrescenta os campos de PLN sem mexer no que veio do site."""
    preparada = dict(noticia)

    texto_limpo = limpar_creditos(noticia["texto"])                  # Parte 2
    texto = normalizar(f"{noticia['titulo']} {texto_limpo}")         # Parte 3

    tokens = tokenizar(texto)                                        # Parte 3
    sem_vazias = tirar_palavras_vazias(tokens)                       # Parte 4
    radicais = achar_radicais(sem_vazias)                            # Parte 5
    lemas = achar_lemas(sem_vazias)

    preparada.update({
        "texto_limpo": texto_limpo,
        "texto_normalizado": texto,
        "palavras": len(texto_limpo.split()),
        "chave": gerar_chave(noticia["titulo"], noticia["texto"]),   # Parte 6
        "tokens": tokens,
        "tokens_sem_vazias": sem_vazias,
        "texto_para_modelo": " ".join(sem_vazias),
        "radicais": radicais,
        "lemas": lemas,
        "texto_radicais": " ".join(radicais),
        "texto_lemas": " ".join(lemas),
    })
    return preparada


preparadas = [preparar_noticia(n) for n in noticias]
base, repetidas, curtas = tirar_repetidas_e_curtas(preparadas)

print(f"{len(noticias)} coletadas")
print(f"{repetidas} repetidas e {curtas} curtas foram tiradas")
print(f"{len(base)} notícias na base final, com {len(base[0])} campos cada\n")

primeira = base[0]
print("EXEMPLO:", primeira["titulo"][:66], "\n")
for campo in ("texto", "texto_limpo", "texto_para_modelo", "texto_radicais", "texto_lemas"):
    print(f"{campo:>18}: {primeira[campo][:88].replace(chr(10), ' | ')}")

# Conferindo a regra mais importante do pipeline:
assert base[0]["texto"] == next(n["texto"] for n in noticias if n["url"] == base[0]["url"])
print("\nO texto original continua intacto depois de todas as etapas.")

---
# Parte 8 — Analisando a base

Antes de usar a base para qualquer coisa, é bom saber o que tem dentro dela e se dá para
confiar. São duas perguntas diferentes.

In [ ]:
tamanhos = [n["palavras"] for n in base]
categorias = Counter(n["categoria"].strip() or "Sem categoria" for n in base)

print(f"notícias ............. {len(base)}")
print(f"categorias ........... {len(categorias)}")
print(f"palavras por notícia . menor {min(tamanhos)}, mediana {median(tamanhos):.0f}, "
      f"média {mean(tamanhos):.0f}, maior {max(tamanhos)}")

vocabulario = {t for n in base for t in n["tokens_sem_vazias"]}
so_radicais = {t for n in base for t in n["radicais"]}
so_lemas = {t for n in base for t in n["lemas"]}
print(f"\npalavras diferentes .. {len(vocabulario)}")
print(f"  viram radicais ..... {len(so_radicais)}")
print(f"  viram lemas ........ {len(so_lemas)}")
print("  (juntar as variações diminui o vocabulário, que é justamente a ideia)")

com_credito = sum(1 for n in base if n["texto_limpo"] != n["texto"])
print(f"\nnotícias que tinham crédito ou propaganda: {com_credito} de {len(base)}")

print("\nNOTÍCIAS POR CATEGORIA")
for categoria, quantas in categorias.most_common():
    print(f"  {categoria:<20} {quantas:>3}  {'#' * quantas}")

In [ ]:
figura, (esquerda, direita) = plt.subplots(1, 2, figsize=(13, 4.5))

nomes = [c for c, _ in categorias.most_common()][::-1]
quantidades = [q for _, q in categorias.most_common()][::-1]
esquerda.barh(nomes, quantidades, color="#2563eb")
esquerda.set_title("Notícias por categoria")

direita.hist(tamanhos, bins=min(20, max(len(base) // 2, 3)), color="#2563eb", edgecolor="white")
direita.set_title("Tamanho das notícias")
direita.set_xlabel("palavras")

plt.tight_layout()
plt.show()

### As palavras que marcam cada categoria

Aqui dá para ver por que valeu a pena fazer tudo o que fizemos antes. Sem tirar os créditos,
`fonte` apareceria em quase toda categoria. Sem tirar as stopwords, o resultado seria `de`,
`que`, `a`.

Na base completa dá para reconhecer o assunto só olhando as palavras: *Ocorrências* fala como
boletim de polícia (`conduzido`, `masculino`, `sofreu`, `bombeiros`) e *Tempo* é quase só
número (`ºc`, `máxima`, `mínima`, `vento`).

In [ ]:
def palavras_mais_comuns(lista, quantas=8, categoria=None):
    """Palavras que mais aparecem, no geral ou numa categoria.

    Ignora número e letra solta, que não dizem nada sobre o assunto.
    """
    contagem = Counter()
    for noticia in lista:
        if categoria and noticia["categoria"].strip() != categoria:
            continue
        for palavra in noticia["tokens_sem_vazias"]:
            if len(palavra) > 1 and not palavra[0].isdigit():
                contagem[palavra] += 1
    return contagem.most_common(quantas)


print("no geral:", ", ".join(p for p, _ in palavras_mais_comuns(base, 12)), "\n")
for categoria, _ in categorias.most_common():
    comuns = palavras_mais_comuns(base, 8, categoria)
    print(f"  {categoria:<20} " + ", ".join(p for p, _ in comuns))

### Dá para confiar na base?

As contagens acima dizem o que tem na base. Agora a gente pergunta outra coisa: será que a
coleta funcionou direito? Cinco conferências, e o esperado é que todas deem zero.

| Conferência | O que significa se der diferente de zero |
|---|---|
| campo vazio | o seletor está errado ou o site mudou de layout |
| data estranha | apareceu um formato de data que a gente não previu |
| endereço repetido | a mesma notícia entrou duas vezes |
| conteúdo repetido | a comparação por hash não funcionou |
| categoria diferente da seção | o site usa subcategoria ou mais de uma categoria |

In [ ]:
def entender_data(texto):
    """Transforma o texto da data em data de verdade. None se não conseguir."""
    texto = (texto or "").strip()
    if not texto:
        return None
    try:
        return datetime.fromisoformat(texto.replace("Z", "+00:00"))
    except ValueError:
        return None


vazios = {c: sum(1 for n in base if not str(n.get(c, "")).strip())
          for c in ("titulo", "data", "categoria", "texto", "url")}
datas = [entender_data(n["data"]) for n in base]
datas_ruins = sum(1 for d in datas if d is None)
enderecos = [n["url"] for n in base]
chaves = [n["chave"] for n in base]

print("CONFERINDO A BASE (esperado: tudo zero)")
print(f"  campos vazios ......... {vazios}")
print(f"  datas estranhas ....... {datas_ruins}")
print(f"  endereços repetidos ... {len(enderecos) - len(set(enderecos))}")
print(f"  conteúdos repetidos ... {len(chaves) - len(set(chaves))}")

boas = [d for d in datas if d]
if boas:
    print(f"  período da base ....... {min(boas).date()} a {max(boas).date()}")

assert not any(vazios.values()), "tem campo vazio na base"
assert datas_ruins == 0, "tem data que não deu para entender"
assert len(enderecos) == len(set(enderecos)), "notícia repetida escapou"
print("\nNenhum problema encontrado.")

# Quantas notícias por mês: mostra que a coleta pega as mais recentes primeiro.
por_mes = Counter(d.strftime("%Y-%m") for d in datas if d)
print("\nNOTÍCIAS POR MÊS")
for mes in sorted(por_mes):
    print(f"  {mes}  {por_mes[mes]:>3}  {'#' * por_mes[mes]}")

In [ ]:
# A categoria que o site declara bate com a seção onde a notícia estava?
print("SEÇÃO ONDE ACHAMOS  x  CATEGORIA QUE O SITE DIZ\n")
cruzamento = Counter(
    (de_onde_veio.get(n["url"], "?"), n["categoria"].strip() or "sem categoria") for n in base
)
diferentes = 0
for (secao, declarada), quantas in sorted(cruzamento.items()):
    # Compara sem acento, para "ocorrencias" bater com "Ocorrências".
    simples = unicodedata.normalize("NFKD", declarada.lower()).encode("ascii", "ignore").decode()
    bate = secao.lower() in simples
    diferentes += 0 if bate else quantas
    print(f"  {secao:<14} -> {declarada:<22} {quantas:>3}{'' if bate else '   <- diferente'}")

print(f"\n{diferentes} de {len(base)} notícias têm categoria diferente da seção.")
print("Acontece quando o site usa uma subcategoria (Oktoberfest dentro de Eventos)")
print("ou marca a notícia com mais de uma categoria ao mesmo tempo.")

### Escolhendo notícias para conferir na mão

Nenhuma conferência automática substitui abrir a notícia no site e comparar. O que o programa
pode fazer é escolher **quais** abrir.

A escolha abaixo não é sorteio: ordena por data e pega de tantas em tantas. Assim a amostra é
sempre a mesma (dá para repetir a conferência depois) e cobre a base inteira, em vez de pegar
só as mais recentes, que costumam ser todas parecidas.

No trabalho completo conferimos 20 notícias assim, comparando título, data, categoria e
parágrafos com a página do site. Resultado: as 20 estavam certas.

In [ ]:
def escolher_amostra(lista, quantidade=5):
    """Escolhe notícias espalhadas pelo período, sempre as mesmas."""
    if quantidade >= len(lista):
        return lista
    ordenadas = sorted(lista, key=lambda n: (n["data"], n["url"]))
    passo = (len(ordenadas) - 1) / (quantidade - 1)
    return [ordenadas[round(i * passo)] for i in range(quantidade)]


print("NOTÍCIAS PARA CONFERIR NO SITE\n")
for n in escolher_amostra(base, 5):
    print(f"  {n['data'][:10]}  [{n['categoria'][:12]:<12}]  {n['titulo'][:54]}")
    print(f"     {n['url']}")

# Rodando de novo tem que dar exatamente as mesmas notícias.
assert [n["url"] for n in escolher_amostra(base, 5)] == \
       [n["url"] for n in escolher_amostra(base, 5)]
print("\nA escolha é sempre a mesma, então dá para repetir a conferência depois.")

---
# Parte 9 — Salvando

Salvamos em **JSONL**: uma notícia por linha, cada linha um JSON. É melhor que um arquivo
JSON único porque dá para ler aos poucos, sem carregar tudo na memória, e dá para acrescentar
linha sem reescrever o arquivo todo.

O `ensure_ascii=False` deixa os acentos legíveis no arquivo. Sem ele, "ação" viraria
`ação`.

In [ ]:
ARQUIVO = "noticias_preparadas.jsonl"

with open(ARQUIVO, "w", encoding="utf-8") as f:
    for noticia in base:
        f.write(json.dumps(noticia, ensure_ascii=False) + "\n")

print(f"{len(base)} notícias salvas em {ARQUIVO}")

# Lendo de volta para conferir que deu certo.
conferindo = [json.loads(linha) for linha in open(ARQUIVO, encoding="utf-8")]
assert len(conferindo) == len(base)
assert conferindo[0]["titulo"] == base[0]["titulo"]
print("Arquivo lido de volta, está tudo certo.\n")

print("campos de cada notícia:")
for campo in conferindo[0]:
    print("  -", campo)

---
# O que dá para fazer com essa base

| Ideia | Campo para usar | Como começar |
|---|---|---|
| Descobrir a categoria automaticamente | `texto_para_modelo` + `categoria` | TF-IDF com Regressão Logística |
| Busca por assunto | `texto_para_modelo` ou `texto_radicais` | BM25, ou TF-IDF com similaridade de cosseno |
| Recomendar notícia parecida | os mesmos vetores da busca | comparar notícia com notícia |
| Achar nomes de pessoas e lugares | `texto` (o original) | spaCy com o modelo `pt_core_news_lg` |
| Resumir e agrupar | `texto`, `chave`, `data` | juntar boletins do mesmo acontecimento |

## O que essa base não consegue responder

1. **Só temos um site.** Não dá para comparar como veículos diferentes contam o mesmo fato,
   nem medir tendência, porque não há com o que comparar.
2. **Pegamos as notícias mais recentes**, não o site inteiro (que tem cerca de 49 mil posts).
3. **Cada notícia tem uma categoria só**, mesmo quando o site usa mais de uma.
4. **Só achamos notícia repetida igualzinha.** Dois boletins de trânsito de dias diferentes,
   com números diferentes, passam como notícias distintas.
5. **Falecidos e Aniversários são listas de nomes** de pessoas comuns. Melhor deixar de fora
   de qualquer coisa que envolva nome de pessoa.
6. **A negação se perde** no `texto_para_modelo`, porque "não" é stopword. Quem precisar dela
   usa o campo `tokens`.

## Rodando o trabalho completo

Este notebook coleta pouca coisa para não demorar. A base do trabalho, com 969 notícias, sai
dos três scripts:

```bash
python coletar.py --paginas 12    # demora bastante, é a coleta de verdade
python preparar.py
python analisar.py
```